In [1]:
pip install yfinance

Note: you may need to restart the kernel to use updated packages.


In [8]:
import yfinance as yf 
import numpy as np
import pandas as pd

tickers = ["AAPL", "MSFT"] # descarga de precios historicos de las acciones.

data = yf.download(tickers, start="2020-01-01", end="2024-01-01")
print(data.columns) # lo ideal es revisar primero el nombre de las columnas.

[*********************100%***********************]  2 of 2 completed

MultiIndex([( 'Close', 'AAPL'),
            ( 'Close', 'MSFT'),
            (  'High', 'AAPL'),
            (  'High', 'MSFT'),
            (   'Low', 'AAPL'),
            (   'Low', 'MSFT'),
            (  'Open', 'AAPL'),
            (  'Open', 'MSFT'),
            ('Volume', 'AAPL'),
            ('Volume', 'MSFT')],
           names=['Price', 'Ticker'])


In [9]:
#adj close es el precio real de la acción, es el dato más puro en finanzas porque elimina el "ruido" que generan los eventos corporativos.
# Mientras que el Close (Precio de Cierre) es simplemente el precio al que terminó la acción en el mercado, 
# el Adj Close recalcula ese precio hacia atrás para reflejar el valor económico real que recibió el inversor
# Cuando una empresa reparte dinero a sus accionistas, ese efectivo sale de la valoración de la empresa. en algunos casos Adj Close no existe,
#entonces se trabaja con Close
precios = data["Close"]

In [10]:
precios

Ticker,AAPL,MSFT
Date,,
2020-01-02,72.400513,152.158417
2020-01-03,71.696625,150.263794
2020-01-06,72.267937,150.652161
2020-01-07,71.928040,149.278564
2020-01-08,73.085114,151.656311
...,...,...
2023-12-22,191.609451,368.236603
2023-12-26,191.065125,368.315308
2023-12-27,191.164093,367.735260


In [11]:
# Los precios solos no sirven de nada, ahora se van a transformar en rendimientos y se calculará el rendimiento porcentual diario
rendimientos = precios.pct_change()

In [12]:
rendimientos # en  este caso es completamente normal que el primer dato tenga NaN, ya que no hay día anterior.

Ticker,AAPL,MSFT
Date,,
2020-01-02,NaN,NaN
2020-01-03,-0.009722,-0.012452
2020-01-06,0.007968,0.002585
2020-01-07,-0.004703,-0.009118
2020-01-08,0.016087,0.015928
...,...,...
2023-12-22,-0.005548,0.002784
2023-12-26,-0.002841,0.000214
2023-12-27,0.000518,-0.001575


In [13]:
rendimientos.head()

Ticker,AAPL,MSFT
Date,,
2020-01-02,NaN,NaN
2020-01-03,-0.009722,-0.012452
2020-01-06,0.007968,0.002585
2020-01-07,-0.004703,-0.009118
2020-01-08,0.016087,0.015928


In [14]:
# para eliminar campos NaN
rendimientos = rendimientos.dropna()

In [16]:
rendimientos.head()

Ticker,AAPL,MSFT
Date,,
2020-01-03,-0.009722,-0.012452
2020-01-06,0.007968,0.002585
2020-01-07,-0.004703,-0.009118
2020-01-08,0.016087,0.015928
2020-01-09,0.021241,0.012493


In [18]:
rendimientos.shape # shape indica las dimensiones

(1005, 2)

In [30]:
rendimientos.mean() # rendimiento promedio diario (esperado) y es el retorno aritmético real

Ticker
AAPL    0.001187
MSFT    0.001095
dtype: float64

In [22]:
rendimientos.std() # mide la volatilidad de las acciones. 

Ticker
AAPL    0.021146
MSFT    0.020546
dtype: float64

In [23]:
rendimientos.corr()

Ticker,AAPL,MSFT
Ticker,,
AAPL,1.000000,0.777003
MSFT,0.777003,1.000000


In [24]:
# Correlación = 0.777, esto indica que se mueven bastante parecido, pero no idéntico 
# +1 → se mueven exactamente igual
# 0 → independientes
# -1 → se mueven en sentido opuesto
# la diversificación es limitada cuando la correlación es alta

In [25]:
#Aunque Apple y Microsoft presentan retornos y volatilidades similares, la alta correlación entre ambos activos (0.77) indica que se 
#comportan de manera muy similar en el mercado. Esto limita significativamente los beneficios de diversificación, por lo que 
#un portafolio compuesto únicamente por estos activos no lograría una reducción sustancial del riesgo.

In [27]:
pesos = np.array([0.5, 0.5]) # aquí se define cuanto dinero se pone a cada acción
portafolio = rendimientos.values @ pesos

np.std(portafolio) # volatilidad del portafolio

np.float64(0.019639856578091804)

In [28]:
# La volatilidad del portafolio se reduce ligeramente debido a la diversificación, 
# sin embargo, la alta correlación entre los activos limita significativamente la magnitud de esta reducción.

In [29]:
#El portafolio construido con Apple y Microsoft presenta una ligera reducción en la volatilidad frente a los activos individuales. 
#Sin embargo, debido a la alta correlación entre ambos, los beneficios de diversificación son limitados, lo que sugiere la necesidad 
#de incluir activos con menor correlación para mejorar el perfil de riesgo.